# Prétraitement et Fusion Multi-Omique (v2)

Ce notebook utilise la classe `ClinicalPreprocessor` pour préparer les données pour la modélisation.

In [5]:
import os
import sys
from pathlib import Path
import pandas as pd
import joblib
import importlib

# Robust path setup
project_root = Path(os.getcwd())
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Racine du projet : {project_root}")

# Import common components
from xai_clinical.data.pancan_loader import PANCANLoader, BRCALoader, PANCANBRCAFusion
# Force reload to ensure the new class is picked up
import xai_clinical.data.preprocessor
importlib.reload(xai_clinical.data.preprocessor)
from xai_clinical.data.preprocessor import ClinicalPreprocessor


AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

## 1. Chargement des Données

In [ ]:
pancan_dir = project_root / "data" / "raw" / "pancan"
brca_dir = project_root / "data" / "raw" / "brca_tcga"

pancan_loader = PANCANLoader(str(pancan_dir))
brca_loader = BRCALoader(str(brca_dir))

clinical_df = brca_loader.get_merged_clinical()
target = brca_loader.extract_survival_target(clinical_df, cutoff_months=24)
expression_df = pancan_loader.load_gene_expression()

fusion = PANCANBRCAFusion(pancan_loader, brca_loader)
fused_df = fusion.fuse_expression_clinical(expression_df, clinical_df, target)

X = fused_df.drop(columns=['TARGET'])
y = fused_df['TARGET']

print(f"Dataset prêt : X={X.shape}, y={y.value_counts().to_dict()}")

## 2. Prétraitement avec ClinicalPreprocessor

On effectue l'imputation, le scaling et la sélection de features.

In [ ]:
preprocessor = ClinicalPreprocessor(n_features=150)

# Split train/test interne au preprocessor ou manuel
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

X_train_proc, y_train_proc = preprocessor.fit_transform(X_train, y_train, X_val, X_test)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"Entraînement : {X_train_proc.shape}")
print(f"Validation : {X_val_proc.shape}")
print(f"Test : {X_test_proc.shape}")

## 3. Sauvegarde des Données Préparées

In [ ]:
processed_dir = project_root / "data" / "processed"
os.makedirs(processed_dir, exist_ok=True)

data_to_save = {
    'X_train': X_train_proc,
    'y_train': y_train_proc,
    'X_val': X_val_proc,
    'y_val': y_val,
    'X_test': X_test_proc,
    'y_test': y_test,
    'feature_names': preprocessor.selected_features
}

joblib.dump(data_to_save, processed_dir / "processed_data_v2.pkl")
joblib.dump(preprocessor, processed_dir / "preprocessor_v2.pkl")

print("Données et préprocesseur sauvegardés.")